# Korean Chatbot - Stage 1 (v2: 라이브러리 토크나이저)
1. 한국어 위키피디아로 사전학습 (토크나이저 = HuggingFace `tokenizers`)
2. KoAlpaca로 파인튜닝

> **v1과의 차이**: 직접 구현한 `BPETokenizer` 대신 HuggingFace `tokenizers`(ByteLevel BPE) 사용.
> 학습이 빠르고(Rust), 공백·줄바꿈 손실이 없으며, 기초 토큰이 256바이트라 merge 예산을 한글 음절에 뺏기지 않는다.
> 산출물은 `tokenizer_hf.json` 으로, v1의 `tokenizer.json` 과 별도 파일이다.

## 0. 환경 설정

In [ ]:
!pip install datasets tqdm tokenizers -q

In [ ]:
import os, sys, shutil, importlib

REPO_URL  = "https://github.com/kkkk2058/korean-chatbot"
REPO_DIR  = "korean-chatbot"
STAGE_DIR = f"{REPO_DIR}/stage1_from_scratch"
DRIVE_DIR = "/content/drive/MyDrive/korean_chatbot"

WIKI_DOCS       = 100_000  # 위키피디아 문서 수 (데이터 확대)
MAX_TRAIN_LINES = None     # 캡 제거: 모은 데이터 전부 사용

# Drive 먼저 마운트 (이어학습/복원이 앞 셀에서 동작하도록)
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL
else:
    !git -C $REPO_DIR pull

sys.path.insert(0, STAGE_DIR)
print("레포 + Drive 준비 완료")

## 1. 토크나이저 (HuggingFace tokenizers · ByteLevel BPE)
라이브러리로 한국어 위키피디아에서 BPE 토크나이저를 학습한다. 캐시가 있으면 로드만 한다.

In [ ]:
import config
from tqdm.notebook import tqdm
from tokenizers import Tokenizer, decoders
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel

TOKENIZER_PATH = "tokenizer_hf.json"
SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]   # id 0,1,2,3 으로 고정 등록

# Drive 캐시에서 가져오기
if not os.path.exists(TOKENIZER_PATH) and os.path.exists(f"{DRIVE_DIR}/tokenizer_hf.json"):
    shutil.copy(f"{DRIVE_DIR}/tokenizer_hf.json", TOKENIZER_PATH)
    print("Drive에서 tokenizer_hf.json 복사 완료")


def load_wiki_texts(n_docs):
    """위키피디아 n_docs개 문서를 스트리밍으로 받아 줄 단위 텍스트 리스트로 반환.
    streaming=True 라 전체(수 GB) 다운로드 없이 앞부분만 받음. (사전학습 셀에서도 재사용)"""
    from datasets import load_dataset
    wiki = load_dataset("wikimedia/wikipedia", "20231101.ko", split="train", streaming=True)
    texts = []
    for i, row in enumerate(tqdm(wiki, total=n_docs, desc="위키 수집")):
        if i >= n_docs:
            break
        for line in row["text"].split("\n"):
            line = line.strip()
            if len(line) > 20:   # 너무 짧은 줄 제외
                texts.append(line)
    return texts


def build_hf_tokenizer():
    """ByteLevel BPE 학습. Rust 구현이라 빠르고, 바이트 단위라 공백/줄바꿈까지 손실 없이 복원된다."""
    tk = Tokenizer(BPE(unk_token="<unk>"))
    tk.pre_tokenizer = ByteLevel(add_prefix_space=False)
    tk.decoder = decoders.ByteLevel()
    trainer = BpeTrainer(vocab_size=config.VOCAB_SIZE, special_tokens=SPECIAL_TOKENS)
    wiki_texts = load_wiki_texts(WIKI_DOCS)
    print(f"코퍼스 문장 수: {len(wiki_texts):,}")
    tk.train_from_iterator(wiki_texts, trainer=trainer)
    return tk


class HFTokenizer:
    """라이브러리 토크나이저를 모델 코드가 기대하는 인터페이스(.vocab/.encode/.decode)로 감싼 어댑터.
    이 덕분에 아래 사전학습/파인튜닝/생성 셀을 v1과 동일하게 쓸 수 있다."""
    def __init__(self, tk):
        self.tk = tk
        self.vocab = tk.get_vocab()          # token(str) -> id(int) 딕셔너리

    def encode(self, text):
        return self.tk.encode(text).ids      # list[int]

    def decode(self, ids):
        return self.tk.decode(ids)           # 특수토큰(<pad>/<s>/</s>) 자동 제외


if os.path.exists(TOKENIZER_PATH):
    tk = Tokenizer.from_file(TOKENIZER_PATH)
    print("기존 tokenizer_hf.json 로드 완료")
else:
    tk = build_hf_tokenizer()
    tk.save(TOKENIZER_PATH)
    shutil.copy(TOKENIZER_PATH, f"{DRIVE_DIR}/tokenizer_hf.json")
    print(f"토크나이저 저장 완료 → {DRIVE_DIR}/tokenizer_hf.json")

tokenizer = HFTokenizer(tk)
print(f"토크나이저 준비 완료. vocab size = {len(tokenizer.vocab)}")

## 2. 모델 초기화

In [ ]:
import torch
from src.model import Transformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = Transformer(
    vocab_size=len(tokenizer.vocab),
    d_model=config.D_MODEL,
    n_heads=config.N_HEADS,
    n_layers=config.N_LAYERS,
    max_seq_len=config.MAX_SEQ_LEN,
    dropout=config.DROPOUT,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"파라미터 수: {total_params:,} ({total_params/1e6:.1f}M)")

## 3. 사전학습 (한국어 위키피디아)
언어 자체를 먼저 학습. 다음 토큰 예측.

**블록 청킹**: 줄 단위 + 패딩 대신, 전체 텍스트를 하나의 토큰 스트림으로 이어붙여
`MAX_SEQ_LEN` 길이로 잘라 쓴다 → 패딩 0%, 문맥 100% 활용 (nanoGPT/GPT-2 방식).

In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split
import time

PAD_ID = tokenizer.vocab["<pad>"]
BOS_ID = tokenizer.vocab["<s>"]
EOS_ID = tokenizer.vocab["</s>"]

BLOCK_SIZE     = config.MAX_SEQ_LEN
PRETRAIN_CACHE = f"{DRIVE_DIR}/pretrain_blocks.pt"   # v2/블록용 캐시 (기존 것과 별도)


class BlockDataset(Dataset):
    """고정 길이 블록 → 다음 토큰 예측. 모든 블록이 같은 길이라 패딩이 필요 없다."""
    def __init__(self, blocks):
        self.blocks = blocks

    def __len__(self):
        return len(self.blocks)

    def __getitem__(self, idx):
        ids = self.blocks[idx]
        return torch.tensor(ids[:-1], dtype=torch.long), torch.tensor(ids[1:], dtype=torch.long)


def build_blocks(texts, tokenizer, block_size, max_lines=None):
    """모든 텍스트를 하나의 토큰 스트림으로 이어붙인 뒤 block_size+1 단위로 자른다.
    줄 단위 패딩과 달리 패딩 0%, 문맥 100% 활용. 문서 경계는 EOS 로 표시."""
    if max_lines:
        texts = texts[:max_lines]
    stream = []
    for text in tqdm(texts, desc="토큰 스트림 생성 중"):
        stream.extend(tokenizer.encode(text))
        stream.append(EOS_ID)
    n = (len(stream) - 1) // block_size          # 끝 자투리는 버림
    blocks = [stream[i * block_size : i * block_size + block_size + 1] for i in range(n)]
    print(f"총 토큰 {len(stream):,} → 블록 {len(blocks):,}개 (블록당 {block_size} 토큰)")
    return blocks


# 토크나이저 셀에서 wiki_texts 를 이미 만들었으면 재사용, 아니면 새로 수집
if "wiki_texts" not in dir():
    wiki_texts = load_wiki_texts(WIKI_DOCS)
cap = "전체" if MAX_TRAIN_LINES is None else f"{MAX_TRAIN_LINES:,}줄"
print(f"위키 텍스트 줄 수: {len(wiki_texts):,}  (변환 대상: {cap})")

t0 = time.time()
if os.path.exists(PRETRAIN_CACHE):
    print("Drive 캐시에서 사전학습 블록 로드 중...")
    blocks = torch.load(PRETRAIN_CACHE)
    print(f"로드 완료 ({time.time()-t0:.1f}초)")
else:
    print("블록 생성 후 Drive에 캐시 저장 (다음 실행부터 즉시 로드)")
    blocks = build_blocks(wiki_texts, tokenizer, block_size=BLOCK_SIZE, max_lines=MAX_TRAIN_LINES)
    torch.save(blocks, PRETRAIN_CACHE)
    print(f"캐시 저장 완료: {PRETRAIN_CACHE} ({time.time()-t0:.1f}초)")

pretrain_dataset = BlockDataset(blocks)
train_size = int(len(pretrain_dataset) * 0.95)
val_size   = len(pretrain_dataset) - train_size
pretrain_train, pretrain_val = random_split(pretrain_dataset, [train_size, val_size])

# 블록이 모두 같은 길이라 collate_fn(패딩) 불필요 → 기본 stacking 사용
pretrain_loader     = DataLoader(pretrain_train, batch_size=config.PRETRAIN_BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
pretrain_val_loader = DataLoader(pretrain_val,   batch_size=config.PRETRAIN_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"사전학습 데이터: train {len(pretrain_train):,} / val {len(pretrain_val):,} / 배치 {len(pretrain_loader):,}")

In [ ]:
import math, time
import matplotlib.pyplot as plt
from torch.cuda.amp import autocast, GradScaler
from IPython.display import display, clear_output

PRETRAIN_CKPT = "pretrain_checkpoint.pt"

criterion  = torch.nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer  = torch.optim.AdamW(model.parameters(), lr=config.PRETRAIN_LR)
scaler     = GradScaler()
total_steps = (len(pretrain_loader) // config.PRETRAIN_GRAD_ACCUM) * config.PRETRAIN_EPOCHS
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

# Drive 체크포인트 복사
if not os.path.exists(PRETRAIN_CKPT) and os.path.exists(f"{DRIVE_DIR}/pretrain_checkpoint.pt"):
    shutil.copy(f"{DRIVE_DIR}/pretrain_checkpoint.pt", PRETRAIN_CKPT)
    print("Drive에서 pretrain_checkpoint.pt 복사 완료")

start_epoch = 0
if os.path.exists(PRETRAIN_CKPT):
    ckpt = torch.load(PRETRAIN_CKPT, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    if "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])
    if "scaler" in ckpt:
        scaler.load_state_dict(ckpt["scaler"])
    start_epoch = ckpt["epoch"] + 1
    print(f"체크포인트 로드. epoch {start_epoch}부터 재개")

history = {"epoch": [], "train": [], "val": []}
train_start = time.time()
fig, ax = plt.subplots(figsize=(8, 4))

def update_plot(title):
    ax.cla()
    ax.plot(history["epoch"], history["train"], "b-o", markersize=4, label="train")
    ax.plot(history["epoch"], history["val"],   "r-o", markersize=4, label="val")
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(True)
    elapsed = time.time() - train_start
    done = len(history["epoch"])
    eta_str = f"  |  ETA {(elapsed/done)*(config.PRETRAIN_EPOCHS - start_epoch - done)/60:.1f}분" if done > 0 else ""
    fig.suptitle(f"경과 {elapsed/60:.1f}분{eta_str}")
    fig.tight_layout(); clear_output(wait=True); display(fig)


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, steps = 0, 0
    pbar = tqdm(loader, desc="train" if train else "val ", leave=False)
    optimizer.zero_grad()

    with torch.set_grad_enabled(train):
        for step, (x, y) in enumerate(pbar):
            x, y = x.to(device), y.to(device)
            with autocast():
                logits = model(x)
                loss   = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
                if train:
                    loss = loss / config.PRETRAIN_GRAD_ACCUM

            if train:
                scaler.scale(loss).backward()
                if (step + 1) % config.PRETRAIN_GRAD_ACCUM == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad()

            total_loss += loss.item() * (config.PRETRAIN_GRAD_ACCUM if train else 1)
            steps += 1
            pbar.set_postfix({"loss": f"{total_loss/steps:.4f}"})

    return total_loss / steps


print("사전학습 시작!")
for epoch in range(start_epoch, config.PRETRAIN_EPOCHS):
    train_loss = run_epoch(pretrain_loader, train=True)
    val_loss   = run_epoch(pretrain_val_loader, train=False)
    elapsed    = time.time() - train_start

    history["epoch"].append(epoch + 1)
    history["train"].append(train_loss)
    history["val"].append(val_loss)
    update_plot("사전학습 Loss")
    print(f"Epoch {epoch+1:02d} | train {train_loss:.4f} | val {val_loss:.4f} | 경과 {elapsed/60:.1f}분")
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
    }, PRETRAIN_CKPT)
    shutil.copy(PRETRAIN_CKPT, f"{DRIVE_DIR}/pretrain_checkpoint.pt")

print("사전학습 완료!")

## 4. 파인튜닝 (KoAlpaca)
사전학습된 모델에 `질문: / 답변:` 형식 학습. 답변 토큰만 loss 계산(질문은 -100 마스킹).

In [ ]:
from datasets import load_dataset

print("KoAlpaca 로드 중...")
koalpaca = load_dataset("beomi/KoAlpaca-v1.1a", split="train")
print(f"샘플 수: {len(koalpaca)}")
print("예시:", koalpaca[0])

In [ ]:
import torch.nn.functional as F

PROMPT_FMT = "질문: {q}\n답변:"


class InstructDataset(Dataset):
    """질문/답변 형식 파인튜닝. 답변 토큰만 loss 계산 (질문은 -100 마스킹)"""
    def __init__(self, hf_dataset, tokenizer, max_seq_len):
        self.samples = []
        for row in tqdm(hf_dataset, desc="파인튜닝 데이터 변환 중"):
            inp = row.get("input", "").strip()
            q   = f"{row['instruction'].strip()}{chr(10)+inp if inp else ''}"
            prompt_ids = [BOS_ID] + tokenizer.encode(PROMPT_FMT.format(q=q))
            answer_ids = tokenizer.encode(" " + row["output"].strip()) + [EOS_ID]
            ids = (prompt_ids + answer_ids)[: max_seq_len + 1]
            if len(ids) > 1:
                self.samples.append((ids, min(len(prompt_ids), len(ids))))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        ids, n_prompt = self.samples[idx]
        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(ids[1:],  dtype=torch.long)
        y[: max(0, n_prompt - 1)] = -100   # 질문 토큰은 학습 제외 (답변만 학습)
        return x, y


def ft_collate_fn(batch):
    xs, ys = zip(*batch)
    max_len = max(x.size(0) for x in xs)
    xs = torch.stack([F.pad(x, (0, max_len - x.size(0)), value=PAD_ID) for x in xs])
    ys = torch.stack([F.pad(y, (0, max_len - y.size(0)), value=-100) for y in ys])  # 패딩도 -100
    return xs, ys


t0 = time.time()
ft_dataset = InstructDataset(koalpaca, tokenizer, max_seq_len=config.MAX_SEQ_LEN)
ft_train_size = int(len(ft_dataset) * 0.9)
ft_val_size   = len(ft_dataset) - ft_train_size
ft_train, ft_val = random_split(ft_dataset, [ft_train_size, ft_val_size])

ft_loader     = DataLoader(ft_train, batch_size=config.FINETUNE_BATCH_SIZE, shuffle=True,  collate_fn=ft_collate_fn, num_workers=2, pin_memory=True)
ft_val_loader = DataLoader(ft_val,   batch_size=config.FINETUNE_BATCH_SIZE, shuffle=False, collate_fn=ft_collate_fn, num_workers=2, pin_memory=True)
print(f"파인튜닝 데이터: train {len(ft_train):,} / val {len(ft_val):,} / 배치 {len(ft_loader):,}  ({time.time()-t0:.1f}초)")

In [ ]:
import os, torch, shutil

# 1) 망가진 파인튜닝 체크포인트 제거 (사전학습부터 새로 파인튜닝)
for p in ["finetune_checkpoint.pt", f"{DRIVE_DIR}/finetune_checkpoint.pt"]:
    if os.path.exists(p):
        os.remove(p); print(f"삭제: {p}")

# 2) 사전학습 모델 가중치 로드
if not os.path.exists("pretrain_checkpoint.pt"):
    shutil.copy(f"{DRIVE_DIR}/pretrain_checkpoint.pt", "pretrain_checkpoint.pt")
model.load_state_dict(torch.load("pretrain_checkpoint.pt", map_location=device)["model"])
print("사전학습 모델 로드 완료")

# 3) 파인튜닝 셋업
FINETUNE_CKPT = "finetune_checkpoint.pt"
FT_LR = 3e-4                                                   # 사전학습과 동일하게 상향
ft_criterion  = torch.nn.CrossEntropyLoss(ignore_index=-100)  # 마스킹(-100) 반영
ft_optimizer  = torch.optim.AdamW(model.parameters(), lr=FT_LR)
ft_scaler     = GradScaler()
ft_total_steps = (len(ft_loader) // config.FINETUNE_GRAD_ACCUM) * config.FINETUNE_EPOCHS
ft_scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(ft_optimizer, T_max=ft_total_steps)
ft_start_epoch = 0

ft_history = {"epoch": [], "train": [], "val": []}
ft_start   = time.time()
fig2, ax2  = plt.subplots(figsize=(8, 4))

def update_ft_plot():
    ax2.cla()
    ax2.plot(ft_history["epoch"], ft_history["train"], "b-o", markersize=4, label="train")
    ax2.plot(ft_history["epoch"], ft_history["val"],   "r-o", markersize=4, label="val")
    ax2.set_title("파인튜닝 Loss"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
    ax2.legend(); ax2.grid(True)
    elapsed, done = time.time() - ft_start, len(ft_history["epoch"])
    eta = f"  |  ETA {(elapsed/done)*(config.FINETUNE_EPOCHS - done)/60:.1f}분" if done else ""
    fig2.suptitle(f"경과 {elapsed/60:.1f}분{eta}")
    fig2.tight_layout(); clear_output(wait=True); display(fig2)


def run_ft_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, steps = 0, 0
    pbar = tqdm(loader, desc="train" if train else "val ", leave=False)
    ft_optimizer.zero_grad()
    with torch.set_grad_enabled(train):
        for step, (x, y) in enumerate(pbar):
            x, y = x.to(device), y.to(device)
            with autocast():
                logits = model(x)
                loss = ft_criterion(logits.view(-1, logits.size(-1)), y.view(-1))
                if train:
                    loss = loss / config.FINETUNE_GRAD_ACCUM
            if train:
                ft_scaler.scale(loss).backward()
                if (step + 1) % config.FINETUNE_GRAD_ACCUM == 0:
                    ft_scaler.unscale_(ft_optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    ft_scaler.step(ft_optimizer); ft_scaler.update()
                    ft_scheduler.step(); ft_optimizer.zero_grad()
            total_loss += loss.item() * (config.FINETUNE_GRAD_ACCUM if train else 1)
            steps += 1
            pbar.set_postfix({"loss": f"{total_loss/steps:.4f}"})
    return total_loss / steps


print("파인튜닝 시작!")
for epoch in range(config.FINETUNE_EPOCHS):
    train_loss = run_ft_epoch(ft_loader, train=True)
    val_loss   = run_ft_epoch(ft_val_loader, train=False)
    ft_history["epoch"].append(epoch + 1)
    ft_history["train"].append(train_loss)
    ft_history["val"].append(val_loss)
    update_ft_plot()
    print(f"Epoch {epoch+1:02d} | train {train_loss:.4f} | val {val_loss:.4f}")
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": ft_optimizer.state_dict(),
        "scheduler": ft_scheduler.state_dict(),
        "scaler": ft_scaler.state_dict(),
    }, FINETUNE_CKPT)
    shutil.copy(FINETUNE_CKPT, f"{DRIVE_DIR}/finetune_checkpoint.pt")

print("파인튜닝 완료!")

## 5. 생성 테스트

In [ ]:
@torch.no_grad()
def generate(prompt, max_new_tokens=100, temperature=0.8, top_k=30, rep_penalty=1.3):
    model.eval()
    text = f"질문: {prompt}\n답변:"
    ids  = [BOS_ID] + tokenizer.encode(text)
    x    = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        if x.size(1) >= config.MAX_SEQ_LEN:
            break
        with autocast():
            logits = model(x)[:, -1, :]
        # 이미 생성된 토큰에 페널티 → 반복 루프 억제
        for tok in set(x[0].tolist()):
            logits[0, tok] /= rep_penalty
        logits = logits / temperature
        topk_vals, _ = torch.topk(logits, top_k)
        logits[logits < topk_vals[:, -1:]] = float("-inf")
        next_id = torch.multinomial(torch.softmax(logits, dim=-1), num_samples=1)
        if next_id.item() == EOS_ID:
            break
        x = torch.cat([x, next_id], dim=1)

    return tokenizer.decode(x[0].tolist()[len(ids):])


prompts = ["한국의 수도는 어디인가요?", "안녕?", "날씨가 덥다", "핸드폰 추천해줘"]
for p in prompts:
    print(f"Q: {p}")
    print(f"A: {generate(p)}")
    print()

## 6. Google Drive 저장

In [ ]:
# Drive는 셀 0에서 이미 마운트됨
os.makedirs(DRIVE_DIR, exist_ok=True)
for fname in ["tokenizer.json", "pretrain_checkpoint.pt", "finetune_checkpoint.pt"]:
    if os.path.exists(fname):
        shutil.copy(fname, f"{DRIVE_DIR}/{fname}")
        print(f"저장 완료: {DRIVE_DIR}/{fname}")